# 日频 Alpha 因子计算

读取 2020–2026 年日行情，筛选当日在市普通股并排除北交所，将价格调整到计算日基准，计算 Alpha 因子。结果按日期写入 `output/年份/日期.parquet`，每行对应一只股票。

预加载仅整理股票资料与行情地址；主循环逐日读取行情，使用有界队列维护历史窗口。

In [1]:
import pandas as pd
from tqdm import tqdm

from TOOL.data import preload, load_market, save_frame, OUTPUT_PATH
from TOOL.Struct import MarketQueue
from TOOL.WQalpha import FIELD_NAMES
from TOOL.factor import factorCal

## 参数与预加载

`years_list` 选择计算年份，`preload` 返回股票资料、交易日期和行情地址。`MarketQueue()` 缓存最近 25 个交易日；各因子按自己的历史长度要求开始计算，历史不足时保留 NaN。

In [2]:
years_list = ['2020', '2021', '2022', '2023', '2024', '2025', '2026']
base_info, dates_list, dates_path_list = preload(years_list)
market = MarketQueue()
print(len(dates_list), dates_list[0], dates_list[-1])

1636 2020-01-02 2026-09-30


## 逐日计算与保存

每日行情先入队，再由 `factorCal` 完成复权、Alpha 计算、非正成交量股票的置空及行顺序对齐。输出包含 `datetime`、`order_book_id` 和 `alpha001` 至 `alpha101`。因子公式见 `TOOL/WQalpha.py`。

In [3]:
for date, date_path in tqdm(list(zip(dates_list, dates_path_list))):
    listed_info = load_market(date, date_path, base_info, FIELD_NAMES)

    #=======
    market.append(date, listed_info)
    result = factorCal(market, listed_info, date)
    save_frame(result, OUTPUT_PATH, date)
    #=======

100%|██████████| 1636/1636 [08:18<00:00,  3.28it/s]


In [4]:
print('保存目录：', OUTPUT_PATH)
print('文件数：', len(list(OUTPUT_PATH.glob('*/*.parquet'))))
result.iloc[:5, :10]

保存目录： /home/wenjy/FactorTrans/WORKFILE/output
文件数： 1636


,datetime,order_book_id,alpha001,alpha002,alpha003,alpha004,alpha005,alpha006,alpha007,alpha008
0,2026-09-30,000001.XSHE,0.334106,-0.227669,0.029204,-0.555556,-0.428417,-0.201375,NaN,-0.407443
1,2026-09-30,000002.XSHE,0.334106,-0.658420,-0.639329,-0.666667,-0.790379,-0.716103,NaN,-0.739105
2,2026-09-30,000006.XSHE,0.045411,-0.126533,-0.751136,-0.333333,-0.330796,-0.572678,NaN,-0.802160
3,2026-09-30,000007.XSHE,-0.197005,0.231558,0.404276,-0.111111,-0.269898,0.153442,NaN,-0.345931
4,2026-09-30,000008.XSHE,0.334106,0.306079,0.096248,-0.666667,-0.411493,-0.473249,NaN,-0.451215
